Шаг 1: Скачивание и подготовка данных

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import joblib

In [8]:
# Читаем CSV-файл
data = pd.read_csv('realty_data.csv')

In [9]:
# Отбор признаков и удаление пропусков
columns_to_use = ['total_square', 'rooms', 'floor']
data = data.dropna(subset=columns_to_use + ['price'])

In [10]:
# Матрица признаков и вектор цели
X = data[columns_to_use].astype(float)
y = data['price'].astype(float)

In [11]:
# Тренировка и тестирование
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [12]:
# Линейная регрессия
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [13]:
# Прогноз и расчет качества
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print(f'Средняя квадратичная ошибка (MSE): {mse:.2f}')

Средняя квадратичная ошибка (MSE): 343857511077308.44


In [14]:
# Сохраняем модель
joblib.dump(model, 'linear_regression_model.pkl')

['linear_regression_model.pkl']

Шаг 2: Создание API с помощью FastAPI

In [15]:
import joblib
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
import pandas as pd

In [16]:
# Загрузка модели
model = joblib.load('linear_regression_model.pkl')

In [17]:
# Инициализация FastAPI
app = FastAPI()

In [19]:
# Класс для хранения данных запроса
class PredictInput(BaseModel):
    total_square: float
    rooms: int
    floor: int

# Health check endpoint
@app.get("/health")
async def health_check():
    """Проверяет состояние API"""
    return {"status": "OK"}

# GET-запрос для прогноза
@app.get("/predict_get/")
async def predict_get(total_square: float, rooms: int, floor: int):
    """Прогнозирует стоимость недвижимости через GET-запрос."""
    input_data = [[total_square, rooms, floor]]
    prediction = model.predict(input_data)[0]
    return {"predicted_price": round(prediction)}

# POST-запрос для прогноза
@app.post("/predict_post/")
async def predict_post(data: PredictInput):
    """Прогнозирует стоимость недвижимости через POST-запрос."""
    input_data = [[data.total_square, data.rooms, data.floor]]
    prediction = model.predict(input_data)[0]
    return {"predicted_price": round(prediction)}

# Запуск API
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop